# M11 · Embeddings & representation learning

_AFP-AI · Domain 2 · Retrieval & Representation_

**Turn messy people, text, and ads into vectors whose geometry can be searched.**

We will build tiny creator and query embeddings, compare dot product with cosine similarity, and measure retrieval recall. The key formula is $\operatorname{cos}(q,i)=\frac{q^\top i}{|q||i|}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## A tiny semantic space

Each creator has four hand-built signals. In production those coordinates are learned, but this toy space lets us inspect the geometry.

In [ ]:
names = np.array(["ai founder", "career coach", "b2b security", "food creator", "cloud architect", "event host"])
emb = np.array([
    [0.90, 0.70, 0.80, 0.10],
    [0.30, 0.85, 0.25, 0.20],
    [0.85, 0.40, 0.95, 0.05],
    [0.05, 0.20, 0.05, 0.95],
    [0.80, 0.35, 0.75, 0.10],
    [0.35, 0.90, 0.30, 0.30],
])
query = np.array([0.88, 0.50, 0.90, 0.05])

print(pd.DataFrame(emb, index=names, columns=["ai", "audience", "security", "lifestyle"]))

## Dot product can reward magnitude

Dot product is useful, but it mixes semantic alignment with vector length. Cosine focuses on direction.

In [ ]:
dot_scores = emb @ query
emb_norm = emb / np.linalg.norm(emb, axis=1, keepdims=True)
query_norm = query / np.linalg.norm(query)
cos_scores = emb_norm @ query_norm

ranking = pd.DataFrame({
    "creator": names,
    "dot": dot_scores,
    "cosine": cos_scores,
})
ranking = ranking.sort_values("cosine", ascending=False)

print(ranking.round(3))

## Step 1 - Normalize and verify

After L2 normalization, every vector should have length $1$. This is the habit that makes cosine retrieval stable.

In [ ]:
lengths = np.linalg.norm(emb_norm, axis=1)
query_length = np.linalg.norm(query_norm)

print(lengths.round(3))
print(round(query_length, 3))

assert np.allclose(lengths, 1.0)
assert np.isclose(query_length, 1.0)

## Step 2 - Compute recall@k

Suppose human judges say three creators are relevant to this brief: AI founder, B2B security, and cloud architect.

In [ ]:
relevant = {"ai founder", "b2b security", "cloud architect"}
top3 = ranking.head(3)["creator"].tolist()
hits = sum(name in relevant for name in top3)
recall_at_3 = hits / len(relevant)

print("top3:", top3)
print("recall@3:", recall_at_3)

assert recall_at_3 >= 2 / 3

## Visualize the scores

A bar chart makes it easy to see which creators the query pulls forward.

In [ ]:
plot_df = ranking.sort_values("cosine")

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(plot_df["creator"], plot_df["cosine"], color="#4c78a8")
ax.set_xlabel("cosine similarity")
ax.set_title("Creator Marketplace semantic retrieval")
plt.show()

## Practice

Try changing the query toward career coaching or food content. Watch how the nearest creators change, then recompute recall with a new relevant set.

In [ ]:
# Your turn:
